# YouTube-bot: Persian Voice + Animated Video — fixed

این نسخه متن فارسی، صدای فارسی واقعی، تصاویر متحرک، متن راست‌به‌چپ و خروجی MP4 می‌سازد. برای صدا ابتدا از صدای فارسی Edge (`fa-IR-DilaraNeural`) استفاده می‌کند؛ اگر در دسترس نبود، به صدای فارسی محلی `espeak-ng` یا صدای fallback می‌رود.

فقط موضوع را تغییر بده و **Runtime → Run all** را بزن.

In [ ]:
#@title تنظیمات
TOPIC = 'چرا آسمان آبی است؟' #@param {type:'string'}
EPISODE_ID = 'persian-animated-001' #@param {type:'string'}
VERTICAL = False #@param {type:'boolean'}
print(TOPIC)

In [ ]:
#@title ساخت و دانلود ویدیو
import subprocess, sys, os, math, random, shutil, zipfile, json, textwrap, asyncio, wave, struct
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','edge-tts','Pillow','arabic-reshaper','python-bidi'],check=True)
subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core espeak-ng'],check=True)
from PIL import Image, ImageDraw, ImageFont
from bidi.algorithm import get_display
import arabic_reshaper
from google.colab import files
root=Path('/content')/EPISODE_ID; shutil.rmtree(root,ignore_errors=True); frames=root/'frames'; frames.mkdir(parents=True)
W,H=(720,1280) if VERTICAL else (1280,720)
script=[f'امروز می خواهیم درباره {TOPIC} صحبت کنیم.',f'شاید پاسخ این سوال ساده به نظر برسد، اما پشت آن یک داستان شگفت انگیز علمی وجود دارد.',f'اگر این سازوکار را مرحله به مرحله ببینیم، موضوع کاملا روشن می شود.',f'حالا می دانیم که {TOPIC} فقط یک واقعیت ساده نیست؛ یک داستان جذاب درباره جهان است.', 'برای داستان های علمی بیشتر، کانال را دنبال کنید.']
fa=lambda s:get_display(arabic_reshaper.reshape(s))
font='/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf'; bold='/usr/share/fonts/truetype/noto/NotoNaskhArabic-Bold.ttf'
if not Path(font).exists(): font='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'; bold=font
tts_text=' '.join(script); audio=root/'narration.mp3'; voice_used='fa-IR-DilaraNeural'
async def make_edge_voice():
    import edge_tts
    await edge_tts.Communicate(tts_text, voice_used, rate='+0%').save(str(audio))
try:
    asyncio.get_event_loop().run_until_complete(make_edge_voice())
except Exception as edge_error:
    print('Edge Persian voice unavailable; trying local espeak-ng:', edge_error)
    wav=root/'narration.wav'; subprocess.run(['espeak-ng','-v','fa','-s','145','-w',str(wav),tts_text],check=False)
    if wav.exists() and wav.stat().st_size>44:
        audio=wav; voice_used='espeak-ng fa'
    else:
        rate=22050; duration_fallback=max(16,int(len(tts_text)/8)); samples=rate*duration_fallback
        with wave.open(str(wav),'w') as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(b''.join(struct.pack('<h',int(1800*math.sin(2*math.pi*220*i/rate))) for i in range(samples)))
        audio=wav; voice_used='audio fallback — replace with Persian recording'
duration=max(16, len(tts_text)/8); fps=12; count=max(1,int(duration*fps)); random.seed(42)
particles=[(random.randrange(W),random.randrange(H),random.randrange(2,8),random.random()*6.28) for _ in range(45)]
for i in range(count):
    t=i/fps; scene=min(len(script)-1,int(t/(duration/len(script)))); img=Image.new('RGB',(W,H),(7,15,45)); d=ImageDraw.Draw(img)
    for y in range(H): d.line((0,y,W,y),fill=(int(8+22*y/H),int(20+45*y/H),int(62+80*y/H)))
    for x,y,r,p in particles:
        xx=int((x+45*math.sin(t*.7+p))%W); yy=int((y+35*math.cos(t*.5+p))%H); d.ellipse((xx-r,yy-r,xx+r,yy+r),fill=(70,170,240))
    d.ellipse((int(W*.68+W*.05*math.sin(t)),int(H*.18+H*.04*math.cos(t)),int(W*.92+W*.05*math.sin(t)),int(H*.42+H*.04*math.cos(t))),outline=(90,220,255),width=5)
    f=ImageFont.truetype(bold,max(30,int(W*.045))); y0=int(H*.48)
    for line in textwrap.wrap(script[scene],width=24 if not VERTICAL else 14):
        shaped=fa(line); box=d.textbbox((0,0),shaped,font=f); x=(W-(box[2]-box[0]))//2; d.text((x+3,y0+3),shaped,font=f,fill=(0,0,0)); d.text((x,y0),shaped,font=f,fill=(245,255,255)); y0+=int(f.size*1.25)
    small=ImageFont.truetype(font,max(18,int(W*.018))); d.text((int(W*.05),int(H*.91)),fa(f'قسمت {scene+1} از {len(script)}  •  Deep Cosmos'),font=small,fill=(180,235,255)); img.save(frames/f'frame-{i:05d}.jpg',quality=90)
video=root/'video.mp4'; cmd=['ffmpeg','-y','-nostdin','-loglevel','error','-framerate',str(fps),'-i',str(frames/'frame-%05d.jpg'),'-i',str(audio),'-c:v','libx264','-c:a','aac','-shortest','-pix_fmt','yuv420p',str(video)]; run=subprocess.run(cmd,text=True,capture_output=True)
if run.returncode!=0 or not video.exists(): print(run.stderr); raise RuntimeError('ساخت ویدیو شکست خورد')
(root/'script-fa.txt').write_text('\n\n'.join(script),encoding='utf-8'); (root/'manifest.json').write_text(json.dumps({'topic':TOPIC,'language':'fa','voice':voice_used,'animated':True,'duration_seconds':duration},ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path('/content')/f'{EPISODE_ID}.zip';
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(root.parent))
files.download(str(archive)); print(f'SUCCESS: {video} | {video.stat().st_size} bytes | voice={voice_used}')